# 분석파이프라인만들기_타이타닉데이터셋_train
* 데이터를 수집 → 정제 → 처리 → 분석 → 결과 생성까지 이어지는 전체 작업 흐름을 자동화·구조화한 과정
* train데이터로 훈련했을 때 사용한 label encoding, scaler, model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib

In [2]:
df = pd.read_csv("./data/Titanic_train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 이전에 확인했던 데이터 특성과 파생변수, 데이터처리 방법을 이용해 파이프라인 구축

In [3]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [4]:
data = df[['Survived', 'Pclass','Sex', 'Age','SibSp','Parch', 'Embarked' ]]
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,0,3,male,22.0,1,0,S
1,1,1,female,38.0,1,0,C
2,1,3,female,26.0,0,0,S
3,1,1,female,35.0,1,0,S
4,0,3,male,35.0,0,0,S
...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,S
887,1,1,female,19.0,0,0,S
888,0,3,female,NaN,1,2,S
889,1,1,male,26.0,0,0,C


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['Survived'])

In [7]:
train

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
692,1,3,male,NaN,0,0,S
481,0,2,male,NaN,0,0,S
527,0,1,male,NaN,0,0,S
855,1,3,female,18.0,0,1,S
801,1,2,female,31.0,1,1,S
...,...,...,...,...,...,...,...
359,1,3,female,NaN,0,0,Q
258,1,1,female,35.0,0,0,C
736,0,3,female,48.0,1,3,S
462,0,1,male,47.0,0,0,S


In [8]:
test

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
565,0,3,male,24.0,2,0,S
160,0,3,male,44.0,0,1,S
553,1,3,male,22.0,0,0,C
860,0,3,male,41.0,2,0,S
241,1,3,female,NaN,1,0,Q
...,...,...,...,...,...,...,...
880,1,2,female,25.0,0,1,S
91,0,3,male,20.0,0,0,S
883,0,2,male,28.0,0,0,S
473,1,2,female,23.0,0,0,C


In [9]:
print(train['Survived'].value_counts())
print(test['Survived'].value_counts())

Survived
0    439
1    273
Name: count, dtype: int64
Survived
0    110
1     69
Name: count, dtype: int64


In [10]:
test.to_csv("./data/titanic_test.csv", index=False)

In [11]:
train['family'] = train['SibSp'] + train['Parch']
train = train.drop(['SibSp', 'Parch'], axis=1)
train

,Survived,Pclass,Sex,Age,Embarked,family
692,1,3,male,NaN,S,0
481,0,2,male,NaN,S,0
527,0,1,male,NaN,S,0
855,1,3,female,18.0,S,1
801,1,2,female,31.0,S,2
...,...,...,...,...,...,...
359,1,3,female,NaN,Q,0
258,1,1,female,35.0,C,0
736,0,3,female,48.0,S,4
462,0,1,male,47.0,S,0


In [12]:
X = train.drop(['Survived'], axis=1)
y = train['Survived']

In [13]:
X

,Pclass,Sex,Age,Embarked,family
692,3,male,NaN,S,0
481,2,male,NaN,S,0
527,1,male,NaN,S,0
855,3,female,18.0,S,1
801,2,female,31.0,S,2
...,...,...,...,...,...
359,3,female,NaN,Q,0
258,1,female,35.0,C,0
736,3,female,48.0,S,4
462,1,male,47.0,S,0


In [14]:
y

692    1
481    0
527    0
855    1
801    1
      ..
359    1
258    1
736    0
462    0
507    1
Name: Survived, Length: 712, dtype: int64

# 범주형 컬럼과 숫자형 컬럼 분리

In [15]:
cat_cols = X.select_dtypes(include = 'object').columns
cat_cols

C:\Users\Admin\AppData\Local\Temp\ipykernel_8296\2334764212.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include = 'object').columns


Index(['Sex', 'Embarked'], dtype='str')

In [16]:
num_cols = X.select_dtypes(exclude = 'object').columns
num_cols

Index(['Pclass', 'Age', 'family'], dtype='str')

# X, y를 train/valid로 나누기

In [17]:
len(X)

712

In [18]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 데이터 전처리 파이프라인 구성
* 수치형 자료: 평균으로 대체 + 스케일링
* 범주형 자료: 최빈값 대체 + 원-핫 인코딩

---

# 📦 Pipeline 사용법 아주 자세한 설명

## 1️⃣ Pipeline은 무엇을 하는가?

Pipeline은 여러 단계를 **순서대로 연결한 하나의 모델 객체**입니다.

```python
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])
```

겉으로 보면 `pipe`는 모델 하나처럼 보이지만,
내부에서는 다음 순서로 동작합니다.

1. preprocess (전처리)
2. model (학습/예측)

---

## 2️⃣ fit()이 호출되면 내부에서 일어나는 일

```python
pipe.fit(X_train, y_train)
```

### 단계별 동작

① preprocess.fit_transform(X_train)

- 수치형 중앙값 계산 (train 기준)
- 평균/표준편차 계산 (train 기준)
- 범주 목록 기억 (train 기준)
- X_train을 숫자 행렬로 변환

② model.fit(X_train_transformed, y_train)

- 변환된 train 데이터로 모델 학습

🔥 핵심: 전처리 통계는 **반드시 train에서만 계산됨**
→ 데이터 누수 방지

---

## 3️⃣ predict()가 호출되면 내부에서 일어나는 일

```python
pipe.predict(X_test)
```

① preprocess.transform(X_test)
- 이미 train에서 학습된 통계값 사용
- test로는 절대 fit하지 않음

② model.predict(X_test_transformed)

🔥 핵심: test는 transform만 수행

---

## 4️⃣ RandomizedSearchCV와 Pipeline

파라미터 이름 규칙:

```
스텝이름__파라미터
```

예시:

```
model__max_depth
model__C
preprocess__cat__ohe__handle_unknown
```

언더바 2개(__)는 “단계 안으로 들어간다”는 의미입니다.

---

## 5️⃣ 왜 Pipeline이 실무에서 중요한가?

✔ 데이터 누수 방지  
✔ 교차검증에서도 fold별로 안전하게 전처리  
✔ 모델 저장 시 전처리 포함 저장 가능  
✔ 서비스에서 raw 데이터 그대로 predict 가능  

---

## 6️⃣ ColumnTransformer + Pipeline 구조 정리

```
원본 데이터
   ↓
ColumnTransformer
   ├─ 수치형: Imputer → Scaler
   └─ 범주형: Imputer → OneHot
   ↓
모델
```

이 전체 공정을 하나로 묶은 것이 Pipeline입니다.

In [19]:
from sklearn.impute import SimpleImputer # 데이터에 결측치를 채워주는 클래스
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer# 컬럼마다 다른 전처리 적용해줌
from sklearn.pipeline import Pipeline

In [20]:
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("ohe", OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ], 
    remainder="drop"
) # 숫자 컬럼은 num_pipe로 전처리하고 문자 컬럼은 cat_pipe로 전처리 한 뒤 나머지 컬럼은 버린다.

In [21]:
print(preprocess)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 Index(['Pclass', 'Age', 'family'], dtype='str')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 Index(['Sex', 'Embarked'], dtype='str'))])


# 모델 정의, 하이퍼파라미터 정의, randomsearch

In [22]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression  # 어떤 값이 특정 범주에 속할 확률을 예측하는 분류 모델
from xgboost import XGBClassifier
from sklearn.svm import SVC 
# 서포트 벡터 머신: 입력 데이터를 보고 결과를 예측하는 지도학습모델 
# (경계선을 만들고 새로운 데이터가 어디에 속하는지 판단하는 모델)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report)

In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 교차검증방법을 어떻게 나눌지 설정 
# (데이터를 클래스 비율을 유지한 채 5등분해서 섞은 뒤 매번 다른 조각을 검증용으로 쓰면서 모델을 평가하겠다.)
scoring = 'roc_auc'

models = {
    "DecisionTree" : DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "RandomForest" : RandomForestClassifier(random_state=42, class_weight="balanced"),
    "LogisticRegression" : LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM" : SVC(probability=True, class_weight='balanced', random_state=42),
    "XGBoost" : XGBClassifier(random_State=42)
}

In [24]:
models

{'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
 'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
 'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=2000),
 'SVM': SVC(class_weight='balanced', probability=True, random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraint

# 모델별 하이퍼파라미터 정의

In [26]:
# model__: Pipeline 안에 있는 model 단계의 파라미터를 건드린다.
params = {
    "DecisionTree" : {
        "model__max_depth": [2,3,4,5,7,10,15,20],     # 트리를 얼마나 복잡하게 쪼갤 것인가 (클수록 복잡)
        "model__min_samples_split" : [2,5,10,20,30],  # 노드를 쪼개기 위해 필요한 최소 데이터 개수 
        "model__min_samples_leaf" : [1,2,5,10,20],    # 마지막 리프 노드가 가져야 하는 최소 데이터 수
        "model__criterion" : ["gini", "entropy", "log_loss"] # 어떤 기준으로 데이터를 나눌지 결정
    },
    "RandomForest" : {
        "model__n_estimators" : [200, 400, 600, 800, 1000],  # 트리를 몇 개 만들 것인가
        "model__criterion" : ["gini", "entropy", "log_loss"],
        "model__min_samples_split" : [2,5,10,20,30],
        "model__min_samples_leaf" : [1,2,5,10,20],
        "model__max_depth": [2,3,4,5,7,10,15,20]
    },
    "LogisticRegression" : {
        "model__C" : np.logspace(-3, 2, 50), # 모델 복잡도 조절
        "model__solver" : ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'], # 최적화 방법 선택
        "model__penalty" : ['l1', 'l2', 'elasticnet'] # 과적합 방지 방식 선택
    },
    "SVM" : {
        "model__C" : np.logspace(-3, 2, 50),
        "model__gamma" : ['scale', 'auto'] + list(np.logspace(-4, 0, 10)), # 한 데이터 포인트를 기준으로 얼마나 가까운 영역까지 신경 쓸건지
        "model__kernel" : ['rbf', 'sigmoid'] # 데이터를 어떤 방식으로 변환해 분리할지 결정
    },
    "XGBoost" : {
        "model__n_estimators" : [200, 400, 600, 800, 1000],
        "model__max_depth": [2,3,4,5,7,10,15,20],
        "model__learning_rate" : list(np.logspace(-3,-0.3,20))  # 각 트리가 이전 오차를 얼마나 강하게 보정할지 조절하는 파라미터 (learning_rate)
    }
}

params.keys()

dict_keys(['DecisionTree', 'RandomForest', 'LogisticRegression', 'SVM', 'XGBoost'])

In [27]:
from time import time

# 학습이 끝난 최적 모델(best_estimator)을 테스트 데이터로 평가해서 여러 성능지표를 한번에 계산하는 함수 
def evaluate_on_test(best_estimator, X_test, y_test):
    pred = best_estimator.predict(X_test) # 모델에게 테스트 데이터를 넣은 후의 최종 분류 결과 (클래스 예측값)

    score = None 
    if hasattr(best_estimator, "predict_proba"): # 확률을 제공하는 모델의 경우 (ex: 로지스틱 회귀, 랜덤포레스트, XGBoost)
        # hasattr(): "이 모델이 predict_proba라는 기능을 가지고 있어? 있으면 확률 계산 가능한 모델이라는 뜻"
        score = best_estimator.predict_proba(X_test)[:, 1] # 우리는 클래스가 1일 확률만 필요하므로 두번째 열만 가져옴
    elif hasattr(best_estimator, "decision_function"): # 확률을 제공하지 않는 모델의 경우 (ex: SVM)
        score = best_estimator.decision_function(X_test) # 결정경계로부터 얼마나 떨어져 있는지

    metrics = {
        "accuracy": accuracy_score(y_test, pred), # 정답과 모델이 예측한 값을 비교
        "precision": precision_score(y_test, pred, zero_division=0), # zero_division: 0으로 나누는 상황이 생겼을 때 어떻게 처리할지 정함
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }

    if score is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, score)
       
    else:
        metrics["roc_auc"] = np.nan

    return metrics

In [39]:
models.items()

dict_items([('DecisionTree', DecisionTreeClassifier(class_weight='balanced', random_state=42)), ('RandomForest', RandomForestClassifier(class_weight='balanced', random_state=42)), ('LogisticRegression', LogisticRegression(class_weight='balanced', max_iter=2000)), ('SVM', SVC(class_weight='balanced', probability=True, random_state=42)), ('XGBoost', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_con

In [28]:
# 여러 모델을 자동으로 튜닝하고 검증 데이터 성능까지 비교해서 최종 결과표를 만듦
results = []
best_search_objects = {}

N_ITER = 30  # 하이퍼파라미터 조합을 30번만 랜덤 샘플링 
RANDOM_STATE = 42  

for name, model in models.items(): # 모델을 하나씩 자동 실험
#     print(name, model)
    print("="*30, name, "="*30)
    
    # 전처리 + 모델학습을 하나로 묶기
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])
    
    # 하이퍼파라미터 랜덤 탐색 설정 
    search = RandomizedSearchCV(
        estimator=pipe, # "모델만 튜닝하지 말고 전처리 + 모델학습까지 포함된 전체 학습 과정을 대상으로 튜닝해라"
        param_distributions=params[name],
        n_iter=N_ITER,
        scoring=scoring,
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
        refit=True) # CV에서 가장 성능 좋았던 파라미터를 찾은 뒤 그 파라미터로 전체 Train 데이터를 다시 학습해라
    
    search.fit(X_train, y_train)
    best_est = search.best_estimator_
    test_metrics = evaluate_on_test(best_est, X_valid, y_valid)
    
    row = {
        "model" : name,
        "best_cv_score(roc-auc)" : search.best_score_,
        **test_metrics,
        "best_params" : search.best_params_
    }
    results.append(row)
    best_search_objects[name] = search
    
    print(f"best_cv_roc_auc = {search.best_score_:.4f} | test roc-auc = {test_metrics['roc_auc']:.4f}")

results_df = pd.DataFrame(results).sort_values(by='roc_auc', ascending=False)
results_df

============================== DecisionTree ==============================
best_cv_roc_auc = 0.8591 | test roc-auc = 0.8556
============================== RandomForest ==============================
best_cv_roc_auc = 0.8736 | test roc-auc = 0.8405
============================== LogisticRegression ==============================


C:\Users\Admin\miniforge3\envs\fintech\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
100 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Admin\miniforge3\envs\fintech\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Admin\miniforge3\envs\fintech\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\miniforge3\envs\fintech\Lib\site-packages\sklearn\pip

best_cv_roc_auc = 0.8579 | test roc-auc = 0.8289
============================== SVM ==============================
best_cv_roc_auc = 0.8620 | test roc-auc = 0.8256
============================== XGBoost ==============================
best_cv_roc_auc = 0.8731 | test roc-auc = 0.8720


C:\Users\Admin\miniforge3\envs\fintech\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:11:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "random_State" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,model,best_cv_score(roc-auc),accuracy,precision,recall,f1,roc_auc,best_params
4,XGBoost,0.873056,0.839161,0.863636,0.690909,0.767677,0.872004,"{'model__n_estimators': 1000, 'model__max_dept..."
0,DecisionTree,0.859101,0.790210,0.692308,0.818182,0.750000,0.855579,"{'model__min_samples_split': 30, 'model__min_s..."
1,RandomForest,0.873568,0.811189,0.759259,0.745455,0.752294,0.840496,"{'model__n_estimators': 1000, 'model__min_samp..."
2,LogisticRegression,0.857920,0.797203,0.709677,0.800000,0.752137,0.828926,"{'model__solver': 'newton-cg', 'model__penalty..."
3,SVM,0.862000,0.804196,0.764706,0.709091,0.735849,0.825620,"{'model__kernel': 'rbf', 'model__gamma': 0.129..."


In [43]:
results_df['best_params'][4]

{'model__n_estimators': 1000,
 'model__max_depth': 2,
 'model__learning_rate': np.float64(0.007122485584859914)}

In [30]:
print(search)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocess',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer()),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
      

In [31]:
print(search.best_estimator_)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                       

# best 모델 저장하기

# 변수/모델/스케일러 등을 저장/로드하는 방법 joblib(직렬화)
* joblib.dump(변수명, 경로 파일명)
* 변수명 = joblib.load(경로 파일명)

In [32]:
results_df

,model,best_cv_score(roc-auc),accuracy,precision,recall,f1,roc_auc,best_params
4,XGBoost,0.873056,0.839161,0.863636,0.690909,0.767677,0.872004,"{'model__n_estimators': 1000, 'model__max_dept..."
0,DecisionTree,0.859101,0.790210,0.692308,0.818182,0.750000,0.855579,"{'model__min_samples_split': 30, 'model__min_s..."
1,RandomForest,0.873568,0.811189,0.759259,0.745455,0.752294,0.840496,"{'model__n_estimators': 1000, 'model__min_samp..."
2,LogisticRegression,0.857920,0.797203,0.709677,0.800000,0.752137,0.828926,"{'model__solver': 'newton-cg', 'model__penalty..."
3,SVM,0.862000,0.804196,0.764706,0.709091,0.735849,0.825620,"{'model__kernel': 'rbf', 'model__gamma': 0.129..."


In [33]:
best_pipe = search.best_estimator_

In [34]:
import joblib

In [44]:
joblib.dump(best_pipe, "./data/titanic_best_model.joblib")
# 튜닝이 끝난 후 가장 성능이 좋았던 모델 (전처리 + 모델포함)을 파일로 저장한다는 뜻

['./data/titanic_best_model.joblib']

In [36]:
print(best_pipe)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Age', 'family'], dtype='str')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                       